In [1]:
import pandas as pd

df = pd.read_csv("../data/raw_justwatch.csv")
print(df.shape)
df.head()

(1810, 12)


,title,type,genre,release_year,imdb_rating,imdb_votes,platform,content_origin,production_country,runtime_min,age_rating,url
0,Dhurandhar: The Revenge,movie,"Action & Adventure, Crime, Mystery & Thriller",2026.0,8.3,62k,JioHotstar,Indian,India,235.0,A,https://www.justwatch.com/in/movie/dhurandhar-...
1,Trikala: Script of God,movie,Mystery & Thriller,2026.0,NaN,NaN,JioHotstar,Indian,India,127.0,A,https://www.justwatch.com/in/movie/trikala-scr...
2,Dhurandhar,movie,"Action & Adventure, Crime, Drama, Mystery & Th...",2025.0,8.3,143k,JioHotstar,Indian,India,214.0,A,https://www.justwatch.com/in/movie/dhurandhar-...
3,Dridam,movie,"Crime, Drama, Mystery & Thriller, Bollywood",2026.0,7.0,3k,JioHotstar,Indian,India,127.0,UA16+,https://www.justwatch.com/in/movie/dridam
4,Mollywood Times,movie,"Comedy, Drama",2026.0,7.5,2k,JioHotstar,Indian,India,150.0,UA16+,https://www.justwatch.com/in/movie/mollywood-t...


In [2]:
print("Missing values per column:")
print(df.isna().sum())
print(f"\nDuplicate rows (same title+platform+type): {df.duplicated(subset=['title','platform','type']).sum()}")
print(f"\nUnique titles: {df['title'].nunique()}")
print(f"\nPlatform counts:\n{df['platform'].value_counts()}")
print(f"\nType counts:\n{df['type'].value_counts()}")

Missing values per column:
title                   0
type                    0
genre                  15
release_year            3
imdb_rating           324
imdb_votes            324
platform                0
content_origin          3
production_country      3
runtime_min           357
age_rating            355
url                     0
dtype: int64

Duplicate rows (same title+platform+type): 3

Unique titles: 1235

Platform counts:
platform
SonyLIV               463
JioHotstar            450
JioCinema             450
Amazon Prime Video    447
Name: count, dtype: int64

Type counts:
type
movie    913
show     897
Name: count, dtype: int64


In [3]:
df = df.drop_duplicates(subset=['title', 'platform', 'type'])
print(f"After dedup: {df.shape}")

print("\nMissing runtime by type:")
print(df[df['runtime_min'].isna()]['type'].value_counts())

print("\nMissing age_rating by type:")
print(df[df['age_rating'].isna()]['type'].value_counts())

After dedup: (1807, 12)

Missing runtime by type:
type
show     288
movie     69
Name: count, dtype: int64

Missing age_rating by type:
type
show     286
movie     69
Name: count, dtype: int64


In [4]:
def clean_genre(g):
    if pd.isna(g):
        return None
    parts = [p.strip() for p in g.split(",")]
    parts = [p for p in parts if p]
    return ", ".join(dict.fromkeys(parts))  # dedupe while keeping order

df['genre'] = df['genre'].apply(clean_genre)
df['genre_list'] = df['genre'].apply(lambda g: g.split(", ") if pd.notna(g) else [])
print(df[['title', 'genre']].head(10))

                     title                                              genre
0  Dhurandhar: The Revenge      Action & Adventure, Crime, Mystery & Thriller
1   Trikala: Script of God                                 Mystery & Thriller
2               Dhurandhar  Action & Adventure, Crime, Drama, Mystery & Th...
3                   Dridam        Crime, Drama, Mystery & Thriller, Bollywood
4          Mollywood Times                                      Comedy, Drama
5           Evil Dead Rise  Horror, Mystery & Thriller, Horror Thriller, D...
6  Spider-Man: No Way Home  Action & Adventure, Fantasy, Science-Fiction, ...
7     Avatar: Fire and Ash  Drama, Fantasy, Science-Fiction, Action Adventure
8                Send Help  Comedy, Horror, Mystery & Thriller, Horror Thr...
9     Fifty Shades of Grey  Drama, Mystery & Thriller, Romance, Romantic T...


In [5]:
print("Rows still missing content_origin:")
print(df[df['content_origin'].isna()][['title', 'production_country']])

df['content_origin'] = df['content_origin'].fillna('Unknown')

Rows still missing content_origin:
                           title production_country
99    Bāhubali 2: The Conclusion                NaN
549   Bāhubali 2: The Conclusion                NaN
1374  Bāhubali 2: The Conclusion                NaN


In [6]:
def guess_language(row):
    genre = row['genre'] if pd.notna(row['genre']) else ""
    if row['content_origin'] == 'Indian':
        if 'Bollywood' in genre:
            return 'Hindi'
        return 'Regional Indian (unspecified)'
    elif row['content_origin'] == 'International':
        return 'English (assumed)'
    return 'Unknown'

df['language_guess'] = df.apply(guess_language, axis=1)
print(df['language_guess'].value_counts())

language_guess
English (assumed)                1268
Regional Indian (unspecified)     284
Hindi                             252
Unknown                             3
Name: count, dtype: int64


In [7]:
CURRENT_YEAR = 2026
df['is_recent'] = df['release_year'] >= (CURRENT_YEAR - 2)
print(df['is_recent'].value_counts())

is_recent
False    1320
True      487
Name: count, dtype: int64


In [8]:
df.to_csv("../data/cleaned_master.csv", index=False)
print(f"Saved cleaned_master.csv with {df.shape[0]} rows, {df.shape[1]} columns")
print(df.columns.tolist())

Saved cleaned_master.csv with 1807 rows, 15 columns
['title', 'type', 'genre', 'release_year', 'imdb_rating', 'imdb_votes', 'platform', 'content_origin', 'production_country', 'runtime_min', 'age_rating', 'url', 'genre_list', 'language_guess', 'is_recent']


In [9]:
print("=== Total titles per platform ===")
print(df['platform'].value_counts())

print("\n=== Movie vs Show split per platform ===")
print(df.groupby(['platform', 'type']).size().unstack(fill_value=0))

=== Total titles per platform ===
platform
SonyLIV               463
JioHotstar            449
JioCinema             449
Amazon Prime Video    446
Name: count, dtype: int64

=== Movie vs Show split per platform ===
type                movie  show
platform                       
Amazon Prime Video    230   216
JioCinema             224   225
JioHotstar            224   225
SonyLIV               232   231


In [10]:
lang_dist = df.groupby(['platform', 'language_guess']).size().unstack(fill_value=0)
print(lang_dist)

print("\n=== % Regional/Hindi (non-English) per platform ===")
lang_pct = lang_dist.div(lang_dist.sum(axis=1), axis=0) * 100
print(lang_pct.round(1))

language_guess      English (assumed)  Hindi  Regional Indian (unspecified)  \
platform                                                                      
Amazon Prime Video                311     64                             71   
JioCinema                         325     62                             61   
JioHotstar                        325     62                             61   
SonyLIV                           307     64                             91   

language_guess      Unknown  
platform                     
Amazon Prime Video        0  
JioCinema                 1  
JioHotstar                1  
SonyLIV                   1  

=== % Regional/Hindi (non-English) per platform ===
language_guess      English (assumed)  Hindi  Regional Indian (unspecified)  \
platform                                                                      
Amazon Prime Video               69.7   14.3                           15.9   
JioCinema                        72.4   13.8           

In [11]:
quality = df.groupby('platform').agg(
    avg_imdb_rating=('imdb_rating', 'mean'),
    titles_with_rating=('imdb_rating', 'count'),
    total_titles=('title', 'count')
).round(2)

pct_7plus = df[df['imdb_rating'] >= 7].groupby('platform').size() / df.groupby('platform')['imdb_rating'].count() * 100
quality['pct_rated_7plus'] = pct_7plus.round(1)

print(quality)

                    avg_imdb_rating  titles_with_rating  total_titles  \
platform                                                                
Amazon Prime Video             7.30                 367           446   
JioCinema                      7.40                 374           449   
JioHotstar                     7.40                 374           449   
SonyLIV                        7.21                 368           463   

                    pct_rated_7plus  
platform                             
Amazon Prime Video             68.4  
JioCinema                      75.4  
JioHotstar                     75.4  
SonyLIV                        62.5  


In [12]:
platform_counts = df.groupby('title')['platform'].nunique()

df['num_platforms'] = df['title'].map(platform_counts)
df['is_exclusive'] = df['num_platforms'] == 1

print("=== Exclusive vs Shared (all titles) ===")
print(df.drop_duplicates('title')['is_exclusive'].value_counts())

print("\n=== Among top-rated titles (7+), exclusive vs shared ===")
top_rated = df[df['imdb_rating'] >= 7].drop_duplicates('title')
print(top_rated['is_exclusive'].value_counts())

print("\n=== Exclusive title count per platform ===")
exclusive_by_platform = df[df['is_exclusive']]['platform'].value_counts()
print(exclusive_by_platform)

=== Exclusive vs Shared (all titles) ===
is_exclusive
True     766
False    469
Name: count, dtype: int64

=== Among top-rated titles (7+), exclusive vs shared ===
is_exclusive
True     390
False    298
Name: count, dtype: int64

=== Exclusive title count per platform ===
platform
SonyLIV               416
Amazon Prime Video    350
Name: count, dtype: int64


In [13]:
hotstar_titles = set(df[df['platform'] == 'JioHotstar']['title'])
jiocinema_titles = set(df[df['platform'] == 'JioCinema']['title'])

overlap = hotstar_titles & jiocinema_titles
only_hotstar = hotstar_titles - jiocinema_titles
only_jiocinema = jiocinema_titles - hotstar_titles

print(f"JioHotstar titles: {len(hotstar_titles)}")
print(f"JioCinema titles: {len(jiocinema_titles)}")
print(f"Shared between both: {len(overlap)}")
print(f"Only on JioHotstar: {len(only_hotstar)}")
print(f"Only on JioCinema: {len(only_jiocinema)}")

print("\nSample titles only on JioHotstar:", list(only_hotstar)[:5])
print("Sample titles only on JioCinema:", list(only_jiocinema)[:5])

JioHotstar titles: 447
JioCinema titles: 447
Shared between both: 447
Only on JioHotstar: 0
Only on JioCinema: 0

Sample titles only on JioHotstar: []
Sample titles only on JioCinema: []


In [14]:
genre_exploded = df.explode('genre_list')
genre_exploded = genre_exploded[genre_exploded['genre_list'].notna() & (genre_exploded['genre_list'] != '')]

overall_genre_pct = genre_exploded['genre_list'].value_counts(normalize=True) * 100
platform_genre_pct = genre_exploded.groupby('platform')['genre_list'].value_counts(normalize=True).unstack(fill_value=0) * 100

top_genres = overall_genre_pct.head(10).index
comparison = platform_genre_pct[top_genres].T
comparison['Overall'] = overall_genre_pct[top_genres]
print(comparison.round(1))

platform            Amazon Prime Video  JioCinema  JioHotstar  SonyLIV  \
genre_list                                                               
Drama                             16.9       15.8        15.8     13.1   
Action & Adventure                 8.8        8.3         8.3      9.2   
Mystery & Thriller                 9.2        8.2         8.2      5.7   
Comedy                             7.4        7.0         7.0      9.2   
Science-Fiction                    5.9        5.6         5.6      7.2   
Crime                              5.1        6.1         6.1      3.1   
Romance                            5.4        4.3         4.3      5.5   
Fantasy                            3.7        4.1         4.1      6.5   
Animation                          3.2        1.7         1.7      9.2   
Bollywood                          3.5        3.3         3.3      3.1   

platform            Overall  
genre_list                   
Drama                  15.3  
Action & Adventure   

In [15]:
freshness = df.groupby('platform')['is_recent'].mean() * 100
print(freshness.round(1))

platform
Amazon Prime Video    38.1
JioCinema             24.9
JioHotstar            24.9
SonyLIV               20.1
Name: is_recent, dtype: float64


In [16]:
origin_dist = df.groupby(['platform', 'content_origin']).size().unstack(fill_value=0)
origin_pct = origin_dist.div(origin_dist.sum(axis=1), axis=0) * 100
print(origin_dist)
print("\n=== % ===")
print(origin_pct.round(1))

content_origin      Indian  International  Unknown
platform                                          
Amazon Prime Video     135            311        0
JioCinema              123            325        1
JioHotstar             123            325        1
SonyLIV                155            307        1

=== % ===
content_origin      Indian  International  Unknown
platform                                          
Amazon Prime Video    30.3           69.7      0.0
JioCinema             27.4           72.4      0.2
JioHotstar            27.4           72.4      0.2
SonyLIV               33.5           66.3      0.2


In [18]:
# Verified current India monthly subscription prices (mid-tier, July 2026):
# JioHotstar/JioCinema Super tier: Rs 149/month (Mobile tier Rs 79, Premium Rs 299)
# Amazon Prime Video: Rs 299/month
# SonyLIV: Rs 299/month
monthly_price = {
    'JioHotstar': 149,
    'JioCinema': 149,
    'Amazon Prime Video': 299,
    'SonyLIV': 299,
}

titles_7plus = df[df['imdb_rating'] >= 7].groupby('platform').size()
cost_per_7plus = pd.Series(monthly_price) / titles_7plus
print("Titles rated 7+:")
print(titles_7plus)
print("\nCost per 7+ rated title (Rs):")
print(cost_per_7plus.round(2))

Titles rated 7+:
platform
Amazon Prime Video    251
JioCinema             282
JioHotstar            282
SonyLIV               230
dtype: int64

Cost per 7+ rated title (Rs):
Amazon Prime Video    1.19
JioCinema             0.53
JioHotstar            0.53
SonyLIV               1.30
dtype: float64


In [19]:
import os
os.makedirs("../data/summaries", exist_ok=True)

# Q1 - volume
df.groupby(['platform', 'type']).size().unstack(fill_value=0).to_csv("../data/summaries/content_volume.csv")

# Q2 - language
lang_dist.to_csv("../data/summaries/language_distribution.csv")

# Q3 - quality
quality.to_csv("../data/summaries/quality_by_platform.csv")

# Q4 - overlap: matrix of shared titles between platform pairs
platforms = df['platform'].unique()
overlap_matrix = pd.DataFrame(index=platforms, columns=platforms, dtype=int)
for p1 in platforms:
    titles_p1 = set(df[df['platform'] == p1]['title'])
    for p2 in platforms:
        titles_p2 = set(df[df['platform'] == p2]['title'])
        overlap_matrix.loc[p1, p2] = len(titles_p1 & titles_p2)
overlap_matrix.to_csv("../data/summaries/platform_overlap_matrix.csv")
print(overlap_matrix)

# Q5 - genre gaps
comparison.to_csv("../data/summaries/genre_gap_analysis.csv")

# Q6/Q7 combined scorecard per platform
scorecard = pd.DataFrame({
    'total_titles': df.groupby('platform')['title'].count(),
    'avg_imdb_rating': df.groupby('platform')['imdb_rating'].mean().round(2),
    'pct_indian': origin_pct['Indian'].round(1),
    'pct_recent': freshness.round(1),
    'pct_exclusive': (df[df['is_exclusive']].groupby('platform').size() / df.groupby('platform').size() * 100).round(1),
})
scorecard.to_csv("../data/summaries/platform_scorecard.csv")
print(scorecard)

# Q8 - cost per title
cost_per_7plus.to_csv("../data/summaries/cost_per_7plus_title.csv")

# Full cleaned dataset also goes here for reference in Power BI
df.to_csv("../data/summaries/full_cleaned_data.csv", index=False)

print("\nAll summary CSVs saved to data/summaries/")

                    JioHotstar  JioCinema  Amazon Prime Video  SonyLIV
JioHotstar               447.0      447.0                74.0     25.0
JioCinema                447.0      447.0                74.0     25.0
Amazon Prime Video        74.0       74.0               446.0     38.0
SonyLIV                   25.0       25.0                38.0    463.0
                    total_titles  avg_imdb_rating  pct_indian  pct_recent  \
platform                                                                    
Amazon Prime Video           446             7.30        30.3        38.1   
JioCinema                    449             7.40        27.4        24.9   
JioHotstar                   449             7.40        27.4        24.9   
SonyLIV                      463             7.21        33.5        20.1   

                    pct_exclusive  
platform                           
Amazon Prime Video           78.5  
JioCinema                     NaN  
JioHotstar                    NaN  
So

In [20]:
scorecard['pct_exclusive'] = scorecard['pct_exclusive'].fillna(0.0)
scorecard.to_csv("../data/summaries/platform_scorecard.csv")
print(scorecard)

                    total_titles  avg_imdb_rating  pct_indian  pct_recent  \
platform                                                                    
Amazon Prime Video           446             7.30        30.3        38.1   
JioCinema                    449             7.40        27.4        24.9   
JioHotstar                   449             7.40        27.4        24.9   
SonyLIV                      463             7.21        33.5        20.1   

                    pct_exclusive  
platform                           
Amazon Prime Video           78.5  
JioCinema                     0.0  
JioHotstar                    0.0  
SonyLIV                      89.8  
